# Customer Details — Data Cleaning

This notebook documents the data-quality assessment, cleaning decisions, validation, and export of the `customer_details` table.

The original `customer_age` column is preserved, while `customer_age_analysis` is created for analysis after handling unrealistic age values.


## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np

cus_det = pd.read_csv("/content/customer_details.csv")


## 2. Initial Data Inspection

In [ ]:
print("Shape:", cus_det.shape)

cus_det.info()
cus_det.head()


## 3. Check for Duplicates and Missing Values

In [ ]:
print("Duplicate rows:", cus_det.duplicated().sum())

print("\nMissing values:")
print(cus_det.isnull().sum())


In [ ]:
cus_det = cus_det.drop_duplicates()


## 4. Inspect Categorical Values

In [ ]:
cus_det["sex"].value_counts()


### Sex Value Cleaning

An unexpected value was found in `sex` and was standardized to `UNKNOWN`.


In [ ]:
cus_det["sex"] = cus_det["sex"].replace({
    "kvkktalepsilindi": "UNKNOWN"
})

cus_det["sex"].value_counts()


## 5. Customer Age — Data Quality Assessment

In [ ]:
cus_det["customer_age"].describe()


In [ ]:
print("Ages between 15 and 80:", cus_det["customer_age"].between(15, 80).sum())
print("Ages between 81 and 150:", cus_det["customer_age"].between(81, 150).sum())
print("Age = 2022:", (cus_det["customer_age"] == 2022).sum())
print("Ages below 15:", (cus_det["customer_age"] < 15).sum())
print("Negative ages:", (cus_det["customer_age"] < 0).sum())


In [ ]:
cus_det.sort_values("customer_age", ascending=False).head(20)


In [ ]:
cus_det.sort_values("customer_age").head(10)


## 6. Investigating Invalid Ages

The `customer_age` column contains several unrealistic values. In particular:

- 2,102 records have an age value of `2022`.
- 4,610 records have ages between `81` and `150`.
- Other records contain ages below `15`, including negative values.

Because removing all invalid records would cause substantial data loss, `tenure` was investigated as a potential source for recovering a reasonable age value.


In [ ]:
suspect = cus_det[
    (cus_det["customer_age"] > 80) &
    (cus_det["customer_age"] <= 150)
]

suspect["tenure"].describe()


In [ ]:
print(
    "Invalid ages (81-150) with a reasonable tenure (15-80):",
    (
        (cus_det["customer_age"] > 80) &
        (cus_det["customer_age"] <= 150) &
        (cus_det["tenure"].between(15, 80))
    ).sum()
)

print(
    "Age = 2022 with a reasonable tenure (15-80):",
    (
        (cus_det["customer_age"] == 2022) &
        (cus_det["tenure"].between(15, 80))
    ).sum()
)


## 7. Age Correction Strategy

A new analytical column named `customer_age_analysis` is created while preserving the original `customer_age`.

Correction rules:

1. If `customer_age` is between `15` and `80`, keep the original age.
2. If `customer_age` is outside the valid range and `tenure` is between `15` and `80`, use `tenure` as the replacement age.
3. If both values are invalid, set the analytical age to `NaN`.

This approach reduces data loss while preserving the original source values.


In [ ]:
cus_det["customer_age_analysis"] = cus_det["customer_age"]


In [ ]:
cus_det.loc[
    (cus_det["customer_age_analysis"] < 15) |
    (cus_det["customer_age_analysis"] > 80),
    "customer_age_analysis"
] = np.nan


In [ ]:
cus_det.loc[
    (cus_det["customer_age_analysis"].isna()) &
    (cus_det["tenure"].between(15, 80)),
    "customer_age_analysis"
] = cus_det["tenure"]


## 8. Validate the Age Correction

In [ ]:
cus_det["customer_age_analysis"].describe()


In [ ]:
print("Missing analytical ages:", cus_det["customer_age_analysis"].isna().sum())
print(
    "Analytical ages outside 15-80:",
    (
        (cus_det["customer_age_analysis"] < 15) |
        (cus_det["customer_age_analysis"] > 80)
    ).sum()
)


In [ ]:
cus_det.loc[
    cus_det["customer_age"] == 2022,
    ["customer_age", "tenure", "customer_age_analysis"]
].head(20)


### Validation Result

After the correction, `customer_age_analysis` contains only values within the selected analytical range (`15–80`) or `NaN` where the age could not be reasonably recovered.


## 9. Final Table Check

In [ ]:
cus_det.info()


In [ ]:
cus_det.isnull().sum()


In [ ]:
cus_det.head()


## 10. Export Clean Dataset

In [ ]:
cus_det.to_csv(
    "customer_details_clean.csv",
    index=False
)


The cleaned customer table is exported as `customer_details_clean.csv`.

The original `customer_age` column is retained for traceability, while `customer_age_analysis` is the age field intended for analysis.
